# Lightweight Fine-Tuning Project

TODO: In this cell, describe your choices for each of the following

* PEFT technique: 
* Model: 
* Evaluation approach: 
* Fine-tuning dataset: 

## Loading and Evaluating a Foundation Model

TODO: In the cells below, load your chosen pre-trained Hugging Face model and evaluate its performance prior to fine-tuning. This step includes loading an appropriate tokenizer and dataset.

In [1]:
# Load the train and test splits of the dair-ai/emotion dataset

from datasets import load_dataset

splits = ['train', 'validation', 'test']

try:
    ds = load_dataset("dair-ai/emotion", "split")
    print("Dataset loaded successfully.")
except Exception as e:
    print(f"An error occurred while loading the dataset: {e}")

#Rducing the dataset to reduce computational resources
try:
    for split in splits:
        ds[split] = ds[split].shuffle(seed=42).select(range(int(0.1*len(ds[split]))))
    print("Data subset created successfully.")
except Exception as e:
    print(f"An error occurred while creating data subset: {e}")

# Show the dataset
ds

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset loaded successfully.
Data subset created successfully.


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 1600
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 200
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 200
    })
})

In [2]:
#Load Tokenizer


from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "openai-community/gpt2"

# Load tokenizer and assign pad_token
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token 
    print("Tokenizer loaded successfully.")
except Exception as e:
    print(f"An error occurred while loading the tokensizer: {e}")


# Load the model AFTER setting pad_token
try:
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=6,
        id2label={0: "sadness", 1: "joy", 2: "love", 3: "anger", 4: "fear", 5: "surprise"},
        label2id={"sadness": 0, "joy": 1, "love": 2, "anger": 3, "fear": 4, "surprise": 5},
    )
    print("Model loaded successfully.")
except Exception as e:
    print(f"An error occurred while loading the Model: {e}")

# Resize the embeddings AFTER model is loaded and tokenizer is set

model.config.pad_token_id = tokenizer.pad_token_id
model.resize_token_embeddings(len(tokenizer))

# (Optional sanity check)
print("Pad token:", tokenizer.pad_token, "| ID:", tokenizer.pad_token_id)



tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenizer loaded successfully.


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at openai-community/gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully.
Pad token: <|endoftext|> | ID: 50256


In [3]:
#Tokenizing data

def preprocess_function(examples):
    """Preprocess the emotion dataset by returning tokenized examples."""
    tokens = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128) # Adjust max_length as needed
    tokens["labels"] = examples["label"]
    return tokens

try:
    tokenized_ds = {}
    for split in splits:
        tokenized_ds[split] = ds[split].map(preprocess_function, batched=True)
    print("Data tokenized successfully.")
except Exception as e:
    print(f"An error while tokenizing data: {e}")

# Show the first example of the tokenized training set
#print(tokenized_ds["train"][0]["input_ids"])

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Data tokenized successfully.


In [5]:
from transformers import DataCollatorWithPadding
try:
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    print("Data collator loaded successfully.")
except Exception as e:
    print(f"An error while loading data collator: {e}")


Data collator loaded successfully.


In [6]:
#Loading Training Arguments
from transformers import TrainingArguments

try:
    training_args = TrainingArguments(
        output_dir="./my_gpt2model", # output directory
        overwrite_output_dir=True,
        num_train_epochs=2, # Number of training epochs
        per_device_train_batch_size=4, # Batch size per device
        weight_decay=0.01,
        per_device_eval_batch_size=4, # Batch size per device
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        learning_rate=5e-5,
    )
    print("Training arguments loaded successfully.")
except Exception as e:
    print(f"An error while loading training arguments: {e}")


Training arguments loaded successfully.


In [7]:
#Setting up trainer
import numpy as np
from transformers import Trainer

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": (predictions == labels).mean()}

try:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds['train'],
        eval_dataset=tokenized_ds['test'],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    print("Trainer loaded successfully.")
except Exception as e:
    print(f"An error while loading trainer: {e}")


Trainer loaded successfully.


In [8]:
trainer.evaluate()

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


{'eval_loss': 10.089971542358398,
 'eval_accuracy': 0.025,
 'eval_runtime': 2.188,
 'eval_samples_per_second': 91.41,
 'eval_steps_per_second': 22.852}

In [10]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.572159,0.825000
2,1.323000,0.448165,0.840000


Checkpoint destination directory ./my_gpt2model/checkpoint-400 already exists and is non-empty.Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./my_gpt2model/checkpoint-800 already exists and is non-empty.Saving will proceed but saved results may be invalid.


TrainOutput(global_step=800, training_loss=1.0391615676879882, metrics={'train_runtime': 133.91, 'train_samples_per_second': 23.897, 'train_steps_per_second': 5.974, 'total_flos': 209044950220800.0, 'train_loss': 1.0391615676879882, 'epoch': 2.0})

In [24]:
import pandas as pd
import numpy as np


df = pd.DataFrame(tokenized_ds["test"])
df = df[["text", "label"]]

# Replace <br /> tags in the text with spaces
df["text"] = df["text"].str.replace("<br />", " ")

# Add the model predictions to the dataframe
predictions = trainer.predict(tokenized_ds["test"])
df["predicted_label"] = np.argmax(predictions[0], axis=1)

df.head(10)

,text,label,predicted_label
0,i was feeling really troubled and down over wh...,0,0
1,i feel so thrilled to have three such distingu...,1,1
2,i feel is that the most likeable characters ar...,1,1
3,i tune out the rest of the world and focus on ...,1,1
4,i sit here writing this i feel unhappy inside,0,0
5,im feeling and if ive liked being pregnant,2,1
6,im very hurt and i feel unimportant,0,0
7,i used to be able to hang around talk with the...,3,4
8,i don t have the feeling of divine vibrations,1,1
9,i vented my feelings towards the pathetic excu...,0,0


## Performing Parameter-Efficient Fine-Tuning

TODO: In the cells below, create a PEFT model from your loaded model, run a training loop, and save the PEFT model weights.

In [11]:
#Setting up LoraConfig
from peft import LoraConfig, TaskType

try:
    config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        #inference_mode=False,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1
    )
    print("Lora configured successfully.")
except Exception as e:
    print(f"An error while configuring Lora: {e}")


Lora configured successfully.


In [12]:
import torch
import torch.nn as nn
from transformers import AutoModelForSequenceClassification

# Load model for PEFT
try:
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=6,   #
    )
    print("Model loaded for PEFT successfully.")
except Exception as e:
    print(f"An error while loading model for PEFT: {e}")

# Fix pad_token_id if necessary
model.config.pad_token_id = tokenizer.pad_token_id

# Configuring model to ensure no. of labels match:
model.config.num_labels = 6
model.config.problem_type = "single_label_classification"
model.classifier = torch.nn.Linear(model.config.hidden_size, 6)

# Test output size if required
inputs = tokenizer("Hello world!", return_tensors="pt").to(model.device)
outputs = model(**inputs)
print(outputs.logits.shape)

print(model.classifier)

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at openai-community/gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded for PEFT successfully.
torch.Size([1, 6])
Linear(in_features=768, out_features=6, bias=True)


In [13]:
# Configuring model for peft
from peft import get_peft_model

try:
    lora_model = get_peft_model(model, config)
    print("Model configured for PEFT successfully.")
except Exception as e:
    print(f"An error while configuring model for PEFT: {e}")
    

/opt/conda/lib/python3.10/site-packages/peft/tuners/lora.py:475: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Model configured for PEFT successfully.


In [14]:
lora_model.print_trainable_parameters()

trainable params: 313,356 || all params: 124,753,164 || trainable%: 0.25118080371893414


In [15]:
lora_model.save_pretrained("pretrained-lora")

In [16]:
#Invoking tokenizer
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


def preprocess_function(examples):
    encoding = tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
    encoding["labels"] = [int(label) for label in examples["label"]]
    return encoding

try:
    tokenized_ds = {}
    for split in splits:
        tokenized_ds[split] = ds[split].map(preprocess_function, batched=True)
    print("Dataset tokenization for PEFT successful.")
except Exception as e:
    print(f"An error while tokenizing PEFT dataset: {e}")


Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Dataset tokenization for PEFT successful.


In [17]:
# Loading Training arguments
from transformers import TrainingArguments

try:
    training_args = TrainingArguments(
        output_dir="./training_loramodel", # output directory
        overwrite_output_dir=True,
        num_train_epochs=2, # Number of training epochs
        per_device_train_batch_size=4, # Batch size per device
        weight_decay=0.01,
        per_device_eval_batch_size=4, # Batch size per device
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        learning_rate=5e-5,
    )
    print("Lora training arguments loaded successfully.")
except Exception as e:
    print(f"An error while loading lora training arguments: {e}")


Lora training arguments loaded successfully.


In [18]:
#Training with lora trainer
import numpy as np
from transformers import Trainer
from transformers import DataCollatorWithPadding

# Create a data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": (predictions == labels).mean()}

try:
    lora_trainer = Trainer(
        model=lora_model,
        args=training_args,
        train_dataset=tokenized_ds['train'],
        eval_dataset=tokenized_ds['test'],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    print("Lora trainer loaded successfully.")
except Exception as e:
    print(f"An error while loading lora trainer: {e}")


Lora trainer loaded successfully.


In [19]:
lora_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.520200,0.405000
2,2.264200,1.491889,0.445000


TrainOutput(global_step=800, training_loss=2.0337374114990237, metrics={'train_runtime': 67.3591, 'train_samples_per_second': 47.507, 'train_steps_per_second': 11.877, 'total_flos': 209803729305600.0, 'train_loss': 2.0337374114990237, 'epoch': 2.0})

In [20]:
lora_trainer.evaluate()

{'eval_loss': 1.4918885231018066,
 'eval_accuracy': 0.445,
 'eval_runtime': 1.8053,
 'eval_samples_per_second': 110.782,
 'eval_steps_per_second': 27.696,
 'epoch': 2.0}

###  ⚠️ IMPORTANT ⚠️

Due to workspace storage constraints, you should not store the model weights in the same directory but rather use `/tmp` to avoid workspace crashes which are irrecoverable.
Ensure you save it in /tmp always.

In [21]:
# Saving the model
lora_model.save_pretrained("posttrained-lora")

## Performing Inference with a PEFT Model

TODO: In the cells below, load the saved PEFT model weights and evaluate the performance of the trained PEFT model. Be sure to compare the results to the results from prior to fine-tuning.

In [22]:
from peft import AutoPeftModelForSequenceClassification
loramodel = AutoPeftModelForSequenceClassification.from_pretrained("posttrained-lora", num_labels=6)

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at openai-community/gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
from transformers import AutoTokenizer
import torch
import torch.nn.functional as F

tokenizer = AutoTokenizer.from_pretrained(model_name)
# Prepare input
inputs = tokenizer("I am feeling very sad", return_tensors="pt")

# Run inference
with torch.no_grad():
    outputs = loramodel(**inputs)
    logits = outputs.logits
    probs = F.softmax(logits, dim=-1)
    predicted_class = torch.argmax(probs, dim=1).item()

print("Class probabilities:", probs)
print("Predicted class:", predicted_class)

Class probabilities: tensor([[0.3950, 0.3083, 0.0238, 0.1216, 0.1197, 0.0316]])
Predicted class: 0


In [24]:
id2label={0: "sadness", 1: "joy", 2: "love", 3: "anger", 4: "fear", 5: "surprise"}
print("Predicted label:", id2label[predicted_class])

Predicted label: sadness


In [9]:
trainer.evaluate()

{'eval_loss': 10.089971542358398,
 'eval_accuracy': 0.025,
 'eval_runtime': 1.6951,
 'eval_samples_per_second': 117.988,
 'eval_steps_per_second': 29.497}

In [25]:
lora_trainer.evaluate()

{'eval_loss': 1.4918885231018066,
 'eval_accuracy': 0.445,
 'eval_runtime': 1.8604,
 'eval_samples_per_second': 107.505,
 'eval_steps_per_second': 26.876,
 'epoch': 2.0}